# 🎮 MC Local AI Bot - Phase 6: Unsloth QLoRA Fine-Tuning (Colab Free T4 / Pro)

This notebook fine-tunes **Qwen 2.5 3B Instruct** on your Minecraft bot dataset using **Unsloth 4-bit QLoRA**.

### ⚡ Why this setup:
- **Super Fast**: Training 350 samples takes ~3-5 minutes on a free Google Colab T4 GPU.
- **0% Local VRAM Usage**: Leaves 100% of your RTX 3060 VRAM free for Minecraft.
- **Turnkey Ollama Export**: Automatically exports to GGUF (`q4_k_m`) and generates `Modelfile` ready for `ollama create mc-qwen:3b`.

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets xformers

## 2. Upload `sft_train.jsonl`
Upload the `sft_train.jsonl` file generated by `export_sft_dataset.py` from your local `data/` folder.

In [ ]:
from google.colab import files
import os

if not os.path.exists('sft_train.jsonl'):
    print('Please upload data/sft_train.jsonl from your local project:')
    uploaded = files.upload()
else:
    print('sft_train.jsonl already present!')

## 3. Load 4-bit Qwen 2.5 3B & Configure LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None  # None for auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)

## 4. Format Dataset with ChatML Template

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": "sft_train.jsonl"})

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(formatted_dataset['train'])} training samples!")

## 5. Fine-Tune with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset["train"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="mc_qwen_lora",
        report_to="none"
    ),
)

trainer_stats = trainer.train()
print("Training Complete!")

## 6. Export to 4-bit GGUF (`q4_k_m`) & Generate Modelfile for Ollama

In [ ]:
# Export to GGUF q4_k_m
model.save_pretrained_gguf("mc-qwen-3b-gguf", tokenizer, quantization_method="q4_k_m")

# Create Modelfile for Ollama
modelfile_content = """FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.1
PARAMETER top_p 0.8
PARAMETER num_gpu 999
"""

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print("Modelfile created!")

## 7. Download Files to Local Machine
Download `unsloth.Q4_K_M.gguf` and `Modelfile` to your local `models/` directory, then run:
```bash
ollama create mc-qwen:3b -f Modelfile
```

In [ ]:
from google.colab import files
import glob

gguf_files = glob.glob("mc-qwen-3b-gguf/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
files.download("Modelfile")